In [9]:
import tensorflow as tf

In [16]:
mode_path = "saved_models/HCRL_model.h5"

In [17]:
model = tf.keras.models.load_model(mode_path)

In [12]:

import numpy as np
import time


# One random sample
sample = np.random.rand(1, *model.input_shape[1:]).astype(np.float32)

# Warm up (important)
for _ in range(20):
    model.predict(sample, verbose=0)

# Measure
N = 1000

start = time.perf_counter()

for _ in range(N):
    model.predict(sample, verbose=0)

end = time.perf_counter()

latency = (end-start)/N

print(f"Average latency: {latency*1000:.3f} ms")
print(f"Throughput: {1/latency:.1f} samples/sec")

Average latency: 59.613 ms
Throughput: 16.8 samples/sec


In [13]:
!pip install psutil


[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import psutil
import os

process = psutil.Process(os.getpid())

before = process.memory_info().rss

_ = model.predict(sample, verbose=0)

after = process.memory_info().rss

print(f"RAM: {(after-before)/1024**2:.2f} MB")

RAM: 0.00 MB


In [18]:
import os

size = os.path.getsize(mode_path)

print(f"Model size = {size/1024:.2f} KB")

Model size = 1510.52 KB


In [19]:
model.summary()

print("Parameters:", model.count_params())

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 30, 8, 32)      │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 15, 4, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 13, 2, 64)      │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 1664)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │       106,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 125,638 (490.78 KB)

 Trainable params: 125,636 (490.77 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

Parameters: 125636


In [20]:
import psutil

print(psutil.cpu_percent(interval=1))

12.4


In [21]:
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

concrete = tf.function(model).get_concrete_function(
    tf.TensorSpec(model.inputs[0].shape, model.inputs[0].dtype)
)

frozen = convert_variables_to_constants_v2(concrete)

graph_def = frozen.graph.as_graph_def()

tf.compat.v1.profiler.profile(
    graph=frozen.graph,
    options=tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
)

Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.
10 ops no flops stats due to incomplete shapes.


name: "_TFProfRoot"
total_definition_count: 1

In [5]:
import tensorflow as tf
import numpy as np
import time
import psutil
import os

def measure_model_performance(model_path, num_samples=1000):
    # -------------------------
    # Load model
    # -------------------------
    model = tf.keras.models.load_model(model_path)

    # Single test sample
    sample = np.random.rand(1, *model.input_shape[1:]).astype(np.float32)

    # -------------------------
    # Model information
    # -------------------------
    model_size = os.path.getsize(model_path) / (1024 * 1024)  # MB
    num_params = model.count_params()

    # -------------------------
    # Memory usage
    # -------------------------
    process = psutil.Process(os.getpid())
    memory_before = process.memory_info().rss

    # Warm-up
    for _ in range(20):
        model.predict(sample, verbose=0)

    memory_after = process.memory_info().rss
    memory_used = (memory_after - memory_before) / (1024 * 1024)

    # -------------------------
    # Inference timing
    # -------------------------
    N = 1000

    times = []

    for _ in range(N):
        start = time.perf_counter()
        model.predict(sample, verbose=0)
        end = time.perf_counter()
        times.append((end - start) * 1000)   # milliseconds

    avg_latency = np.mean(times)
    std_latency = np.std(times)
    min_latency = np.min(times)
    max_latency = np.max(times)
    p95_latency = np.percentile(times, 95)

    throughput = 1000 / avg_latency

    # -------------------------
    # CPU usage
    # -------------------------
    cpu_usage = psutil.cpu_percent(interval=1)

    # -------------------------
    # Summary
    # -------------------------
    print("\n" + "=" * 45)
    print(f" TensorFlow Model Performance Summary of '{model_path}'")
    print("=" * 45)

    print(f"Model size           : {model_size:.2f} MB")
    print(f"Parameters           : {num_params:,}")

    print(f"\nInference latency")
    print(f"  Average            : {avg_latency:.3f} ms")
    print(f"  Std Dev            : {std_latency:.3f} ms")
    print(f"  Minimum            : {min_latency:.3f} ms")
    print(f"  Maximum            : {max_latency:.3f} ms")
    print(f"  95th Percentile    : {p95_latency:.3f} ms")

    print(f"\nThroughput           : {throughput:.1f} samples/sec")
    print(f"Extra RAM used       : {memory_used:.2f} MB")
    print(f"CPU Usage            : {cpu_usage:.1f}%")

    print("=" * 45)

In [7]:
measure_model_performance("CNN/saved_models/HCRL_model.h5")
measure_model_performance("CNN/saved_models/road_model.keras")
measure_model_performance("MLP/saved_models/HCRL_model.h5")


 TensorFlow Model Performance Summary of 'CNN/saved_models/HCRL_model.h5'
Model size           : 1.48 MB
Parameters           : 125,636

Inference latency
  Average            : 59.691 ms
  Std Dev            : 3.977 ms
  Minimum            : 56.668 ms
  Maximum            : 128.875 ms
  95th Percentile    : 61.673 ms

Throughput           : 16.8 samples/sec
Extra RAM used       : 0.03 MB
CPU Usage            : 15.7%



 TensorFlow Model Performance Summary of 'CNN/saved_models/road_model.keras'
Model size           : 2.97 MB
Parameters           : 256,578

Inference latency
  Average            : 59.928 ms
  Std Dev            : 6.187 ms
  Minimum            : 57.494 ms
  Maximum            : 207.417 ms
  95th Percentile    : 61.771 ms

Throughput           : 16.7 samples/sec
Extra RAM used       : 0.57 MB
CPU Usage            : 9.1%

 TensorFlow Model Performance Summary of 'MLP/saved_models/HCRL_model.h5'
Model size           : 0.12 MB
Parameters           : 7,396

Inference latency
  Average            : 59.829 ms
  Std Dev            : 4.337 ms
  Minimum            : 56.572 ms
  Maximum            : 137.713 ms
  95th Percentile    : 62.313 ms

Throughput           : 16.7 samples/sec
Extra RAM used       : 0.24 MB
CPU Usage            : 3.6%
